# Predição: Imagens de mergulho com o modelo MSO-V1

Autor:  Viviane da Rosa Sommer

Atualizado: 14/11/2024

## Objetivo:
Notebook para fazer a predição das imagens de mergulho pelo modelo MSO-V1, para analisar o desempenho do mesmo.

## Importações Necessárias

In [ ]:
!pip install ultralytics
!pip install opencv-python-headless
!pip install torch

import glob
import cv2
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os 
import torchvision
import json
import torch
import zipfile

## Declaração de Constantes e Modelos

In [ ]:
CORAL_CLASS_ID = 0
RESIZED_DIM_CORAL = 800

INPUT_DIRECTORY_POSITIVE_IMAGES = "Lote 4 e 5/Imagens-Mergulho/Coral Mergulho/POSITIVO"
INPUT_DIRECTORY_NEGATIVE_IMAGES = "Lote 4 e 5/Imagens-Mergulho/Coral Mergulho/NEGATIVO"

OUTPUT_DIRECTORY = "pred_mso_v1_mergulho"
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

coral_model = YOLO('runs/segment/Lote123-SOverlay/train-800-lote123/weights/best.pt')

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> tuple:
    """
    Reads an image from a file and returns it along with its dimensions.

    Parameters:
        filename (str): Path to the image file.

    Returns:
        tuple: A tuple containing:
            - img (np.ndarray): The loaded image in BGR format.
            - base_height (int): Height of the image.
            - base_width (int): Width of the image.
            If the image cannot be loaded, returns (None, None, None).
    """
    img = cv2.imread(filename)
    if img is None:
        print(f"Failed to load image: {filename}")
        return None, None, None
        
    base_height, base_width = img.shape[:2]
    return img, base_height, base_width




def process_results(results: list) -> any:
    """
    Processes the results obtained from the model, returning the first valid result with masks.

    Parameters:
        results (list): List of model detection results.

    Returns:
        any: The first valid result containing masks, or None if no valid result is found.
    """
    for result in results:
        if result.masks is not None:
            return result
    return None


def generate_mask_from_polygon(polygon: list, height: int, width: int) -> np.ndarray:
    """
    Generates a binary mask from a given polygon.

    Parameters:
        polygon (list): List of (x, y) coordinates defining the vertices of the polygon.
        height (int): Height of the mask.
        width (int): Width of the mask.

    Returns:
        np.ndarray: A binary mask of shape (height, width), with the polygon area filled with 1s.
    """
    mask = np.zeros((height, width), dtype=np.uint8)
    polygon = np.array(polygon, dtype=np.int32)  
    cv2.fillPoly(mask, [polygon], 1) 
    return mask


def zip_directories(directories: list, zip_file_name: str) -> None:
    """
    Compresses specified directories into a ZIP file.

    Parameters:
        directories (list): List of directory paths to compress.
        zip_file_name (str): Name of the resulting ZIP file.

    Returns:
        None
    """
    with zipfile.ZipFile(zip_file_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for directory in directories:
            root_folder_name = os.path.basename(directory.rstrip('/\\'))  
            for root, _, files in os.walk(directory):
                for file in files:
                    full_path = os.path.join(root, file)
                    relative_path = os.path.relpath(full_path, directory)
                    unique_relative_path = os.path.join(root_folder_name, relative_path)
                    zipf.write(full_path, arcname=unique_relative_path)
                    
                    
def prediction_coral(img: np.array) -> tuple:
    """
    Predicts coral objects using the coral model and returns a binary mask for coral regions.

    Parameters:
        img (np.array): Input image for prediction.

    Returns:
        tuple: A tuple containing:
            - centered_mask (np.ndarray): Resized coral mask with centered annotations.
            - torch.Tensor: The centered mask as a PyTorch tensor.
    """
    original_height, original_width = img.shape[:2]
    coral_results = coral_model(img, verbose=False)
    coral_best_result = process_results(coral_results)
    
    if coral_best_result is not None and coral_best_result.masks is not None:
        masks = coral_best_result.masks.data
        boxes = coral_best_result.boxes.data
        classes = boxes[:, 5]
        scores = boxes[:, 4]

        coral_indices = torch.where((classes == CORAL_CLASS_ID) & (scores > 0.5))[0]
        coral_boxes = boxes[coral_indices]
        coral_masks = masks[coral_indices]
        coral_scores = scores[coral_indices]

        if len(coral_boxes) > 0:
            nms_indices = torchvision.ops.nms(coral_boxes[:, :4], coral_scores, iou_threshold=0.2)
            coral_boxes = coral_boxes[nms_indices]
            coral_masks = coral_masks[nms_indices]
            final_coral_mask = torch.any(coral_masks, dim=0).int() * 255
        else:
            final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    else:
        final_coral_mask = torch.zeros((original_height, original_width), dtype=torch.uint8)
    
    final_coral_mask_np = final_coral_mask.cpu().numpy()
    mask_height, mask_width = final_coral_mask_np.shape

    resized_final_coral_mask = cv2.resize(final_coral_mask_np, (original_width, original_height), interpolation=cv2.INTER_NEAREST)
    centered_mask = np.zeros((original_height, original_width), dtype=np.uint8)
    y_offset = (original_height - resized_final_coral_mask.shape[0]) // 2
    x_offset = (original_width - resized_final_coral_mask.shape[1]) // 2
    if (y_offset >= 0 and x_offset >= 0 and 
        y_offset + resized_final_coral_mask.shape[0] <= original_height and 
        x_offset + resized_final_coral_mask.shape[1] <= original_width):
        
        centered_mask[y_offset:y_offset + resized_final_coral_mask.shape[0], 
                      x_offset:x_offset + resized_final_coral_mask.shape[1]] = resized_final_coral_mask
    else:
        centered_mask = resized_final_coral_mask.copy()

    return centered_mask, torch.from_numpy(centered_mask).int()


def read_annotation_file(filename: str) -> torch.Tensor:
    """
    Reads an annotation file to generate a binary mask for the coral object.

    Parameters:
        filename (str): The filename of the image (used to locate the corresponding annotation file).

    Returns:
        torch.Tensor: A binary mask (0 or 255 values) of the coral annotation.
    """
    if os.path.exists(filename.replace("jpg", "json")):
        with open(filename.replace("jpg", "json"), 'r') as f:
            annotation_data = json.load(f)
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
        for shape in annotation_data['shapes']:
            polygon = shape['points']
            mask = generate_mask_from_polygon(polygon, base_height, base_width)
            mask_tensor = torch.from_numpy(mask)
            annotated_coral_mask = annotated_coral_mask | mask_tensor
    else:
        annotated_coral_mask = torch.zeros((base_height, base_width), dtype=torch.uint8)
    annotated_coral_mask *= 255
    return annotated_coral_mask


def save_images(img: np.ndarray, 
                OUTPUT_DIRECTORY: str,
                intersection_mask: np.ndarray) -> None:
    """
    Saves and displays annotated images, comparing expert and model annotations.

    Parameters:
        img (np.ndarray): The original image in BGR format.
        OUTPUT_DIRECTORY (str): Directory where the annotated image will be saved.
        intersection_mask (np.ndarray): Mask indicating intersections between model and expert annotations.

    This function generates two side-by-side plots:
    1. **Expert Annotation**: Displays expert annotation contours overlaid on the original image in green.
    2. **Model Annotation**: Displays model-detected contours in red, with intersections overlaid.

    The annotated image is saved as a PNG file in the specified directory with a filename based on the input image name.
    """
    filename_simple = filename.split("/")[-1].split(".")[0]

    annotated_img = img.copy()
    if intersection_mask.any():
        contours, _ = cv2.findContours(intersection_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(annotated_img, contours, -1, (0, 0, 255), thickness=4) 

    title = "Positive" if os.path.basename(os.path.dirname(filename)) != "NEGATIVO" else "Negative"
    plt.figure(figsize=(10, 7))
    plt.title(f'Model Annotation - {title}')
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()

    filename_simple = os.path.splitext(os.path.basename(filename))[0]
    plt.savefig(os.path.join(OUTPUT_DIRECTORY, f"density_{filename_simple}.png"))
    plt.show()
    plt.close()
        

## Processamento das imagens - Casos Positivos

In [ ]:
for i, filename in enumerate(glob.glob(f"{INPUT_DIRECTORY_POSITIVE_IMAGES}/**")):
    try:
        if filename.split("/")[-1].split(".")[-1] in ["jpg","png","JPG"]:
            img, base_height, base_width = read_image(filename)
            if img is None:
                continue

            coral_mask_resized, coral_mask_tensor = prediction_coral(img)
            save_images(img, OUTPUT_DIRECTORY, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue


## Processamento das imagens, salvando os resultados - Casos Negativos

In [ ]:
for i, filename in enumerate(glob.glob(f"{INPUT_DIRECTORY_NEGATIVE_IMAGES}/**")):
    try:
        if filename.split("/")[-1].split(".")[-1] in ["jpg","png","JPG"]:
            img, base_height, base_width = read_image(filename)
            if img is None:
                continue

            coral_mask_resized, coral_mask_tensor = prediction_coral(img)
            save_images(img, OUTPUT_DIRECTORY, coral_mask_resized)
    except Exception as e:
        print(f"Error processing file {filename}: {e}")
        continue
